<a href="https://colab.research.google.com/github/PreethamHD/DP-MMFL/blob/main/notebooks/03_create_experiment_splits.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import sys
from pathlib import Path
from google.colab import drive, userdata

# 1. Mount persistent Google Drive
drive.mount("/content/drive")

# 2. Configure Git identity and remote access
GITHUB_USERNAME = "PreethamHD"
REPO_NAME = "DP-MMFL"

!git config --global user.name "PreethamHD"
!git config --global user.email "preethamgowda837@gmail.com"

try:
    token = userdata.get("GITHUB_TOKEN")
    repo_url = f"https://{token}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
except Exception:
    repo_url = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

# 3. Clone if not present, or pull latest changes
%cd /content
if not os.path.exists(f"/content/{REPO_NAME}"):
    !git clone {repo_url}
    %cd /content/{REPO_NAME}
else:
    %cd /content/{REPO_NAME}
    !git pull origin main

# 4. Expose src module to Python path
SRC_PATH = f"/content/{REPO_NAME}/src"
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

# 5. Install required base libraries
!pip install -q pyarrow

print("=" * 50)
print("Workspace setup complete.")
print(f"Working Directory: {os.getcwd()}")
print(f"Drive Root:        /content/drive/MyDrive/{REPO_NAME}")
print("=" * 50)

Mounted at /content/drive
/content
Cloning into 'DP-MMFL'...
remote: Enumerating objects: 76, done.
remote: Counting objects: 100% (76/76), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 76 (delta 23), reused 56 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (76/76), 43.43 KiB | 14.48 MiB/s, done.
Resolving deltas: 100% (23/23), done.
/content/DP-MMFL
Workspace setup complete.
Working Directory: /content/DP-MMFL
Drive Root:        /content/drive/MyDrive/DP-MMFL


In [2]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit

# Ensure repo module visibility
REPO_SRC = "/content/DP-MMFL/src"
if REPO_SRC not in sys.path:
    sys.path.append(REPO_SRC)

PROJECT_ROOT = Path("/content/drive/MyDrive/DP-MMFL")
MANIFEST_PATH = PROJECT_ROOT / "data" / "processed" / "chexpert_plus_manifest.parquet"
SPLIT_MANIFEST_PATH = PROJECT_ROOT / "data" / "processed" / "chexpert_plus_manifest_v2.parquet"

manifest = pd.read_parquet(MANIFEST_PATH)
print("Loaded original manifest shape:", manifest.shape)

Loaded original manifest shape: (223462, 50)


In [3]:
official_train = manifest[manifest["split"] == "train"].copy()
official_valid = manifest[manifest["split"] == "valid"].copy()

print(f"Official train records:  {official_train.shape[0]:,}")
print(f"Official valid records:  {official_valid.shape[0]:,}")
print(f"Official train patients: {official_train['deid_patient_id'].nunique():,}")
print(f"Official valid patients: {official_valid['deid_patient_id'].nunique():,}")

assert official_train.shape[0] == 223228, "Train count mismatch against CheXpert Plus standard."
assert official_valid.shape[0] == 234, "Valid count mismatch against CheXpert Plus standard."

Official train records:  223,228
Official valid records:  234
Official train patients: 64,525
Official valid patients: 200


In [4]:
# Stage 1: Split official train into 80% train and 20% temporary holdout
gss_outer = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, temp_idx = next(
    gss_outer.split(official_train, groups=official_train["deid_patient_id"])
)

dev_train = official_train.iloc[train_idx].copy()
temp_holdout = official_train.iloc[temp_idx].copy()

# Stage 2: Split 20% holdout equally into validation (10%) and test (10%)
gss_inner = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_idx, test_idx = next(
    gss_inner.split(temp_holdout, groups=temp_holdout["deid_patient_id"])
)

dev_val = temp_holdout.iloc[val_idx].copy()
dev_test = temp_holdout.iloc[test_idx].copy()

#Tag partitions
dev_train["experiment_split"] = "train"
dev_val["experiment_split"] = "val"
dev_test["experiment_split"] = "test"
official_valid["experiment_split"] = "official_valid"

manifest_v2 = pd.concat(
    [dev_train, dev_val, dev_test, official_valid],
    ignore_index=True
)

print("\n--- Partition Volumes ---")
for split_name in ["train", "val", "test", "official_valid"]:
    subset = manifest_v2[manifest_v2["experiment_split"] == split_name]
    print(
        f"{split_name:<15}: {len(subset):>7,} images | "
        f"{subset['deid_patient_id'].nunique():>6,} unique patients"
    )


--- Partition Volumes ---
train          : 178,908 images | 51,620 unique patients
val            :  21,757 images |  6,452 unique patients
test           :  22,563 images |  6,453 unique patients
official_valid :     234 images |    200 unique patients


In [5]:
# Extracting unique patient sets across every assigned split
split_patients = {
    split: set(manifest_v2.loc[manifest_v2["experiment_split"] == split, "deid_patient_id"])
    for split in ["train", "val", "test", "official_valid"]
}

#Pairwise overlap assertion
splits = list(split_patients.keys())
total_leakage = 0

print("--- Pairwise Patient Overlap Checks (Target: 0) ---")
for i in range(len(splits)):
    for j in range(i + 1, len(splits)):
        a, b = splits[i], splits[j]
        overlap = split_patients[a] & split_patients[b]
        overlap_size = len(overlap)
        total_leakage += overlap_size
        print(f"{a:<14} vs {b:<14}: {overlap_size} overlaps")

assert total_leakage == 0, f"DATA LEAKAGE DETECTED: {total_leakage} cross-split patient intersections."

#Ensuring each patient maps to exactly one assignment
patient_split_multiplicity = manifest_v2.groupby("deid_patient_id")["experiment_split"].nunique()
assert (patient_split_multiplicity == 1).all(), "Patients discovered with multi-split assignments."

print("\nMutual exclusivity passed: All 64,725 patients exist in exactly one partition.")

--- Pairwise Patient Overlap Checks (Target: 0) ---
train          vs val           : 0 overlaps
train          vs test          : 0 overlaps
train          vs official_valid: 0 overlaps
val            vs test          : 0 overlaps
val            vs official_valid: 0 overlaps
test           vs official_valid: 0 overlaps

Mutual exclusivity passed: All 64,725 patients exist in exactly one partition.


In [6]:
TARGET_COLUMNS = [
    "Enlarged Cardiomediastinum",
    "Cardiomegaly",
    "Lung Opacity",
    "Lung Lesion",
    "Edema",
    "Consolidation",
    "Pneumonia",
    "Atelectasis",
    "Pneumothorax",
    "Pleural Effusion",
    "Pleural Other",
    "Fracture",
    "Support Devices",
]

records = []
for split_name in ["train", "val", "test", "official_valid"]:
    sub = manifest_v2[manifest_v2["experiment_split"] == split_name]
    for label in TARGET_COLUMNS:
        mask = sub[f"mask_{label}"] == 1
        target = sub.loc[mask, f"target_{label}"]

        sup_count = int(mask.sum())
        pos_count = int((target == 1.0).sum())
        rate = (pos_count / sup_count) if sup_count > 0 else np.nan

        records.append({
            "split": split_name,
            "label": label,
            "supervised": sup_count,
            "positive": pos_count,
            "prevalence": round(rate, 4) if not np.isnan(rate) else None
        })

split_stats_df = pd.DataFrame(records)

# Reshape into a comparative view across splits
prevalence_table = split_stats_df.pivot(
    index="label",
    columns="split",
    values="prevalence"
)[["train", "val", "test", "official_valid"]]

print("--- Positive Prevalence by Partition ---")
print(prevalence_table.to_string())

--- Positive Prevalence by Partition ---
split                        train     val    test  official_valid
label                                                             
Atelectasis                 0.9817  0.9789  0.9775          0.9167
Cardiomegaly                0.5790  0.6029  0.5840          0.5397
Consolidation               0.3072  0.3051  0.3046          0.2037
Edema                       0.7020  0.7071  0.6943          0.6463
Enlarged Cardiomediastinum  0.2187  0.2298  0.2046          0.2286
Fracture                    0.7069  0.7016  0.7231          0.6667
Lung Lesion                 0.9007  0.8982  0.8883          1.0000
Lung Opacity                0.9453  0.9449  0.9391          0.9386
Pleural Effusion            0.6819  0.6840  0.6775          0.5962
Pleural Other               0.9905  0.9964  0.9935          1.0000
Pneumonia                   0.5573  0.5714  0.5514          0.8000
Pneumothorax                0.2219  0.2208  0.2205          0.2135
Support Devices      

In [7]:
# Write to persistent Drive storage
manifest_v2.to_parquet(SPLIT_MANIFEST_PATH, index=False)
print(f"Persisted manifest_v2 to: {SPLIT_MANIFEST_PATH}")

# Reload to verify schema and partition counts
verified_df = pd.read_parquet(SPLIT_MANIFEST_PATH)
assert verified_df.shape == manifest_v2.shape, "Shape mismatch after reload."
assert "experiment_split" in verified_df.columns, "experiment_split column missing."

print("Integrity check complete. Breakdown:")
print(verified_df["experiment_split"].value_counts(dropna=False))

Persisted manifest_v2 to: /content/drive/MyDrive/DP-MMFL/data/processed/chexpert_plus_manifest_v2.parquet
Integrity check complete. Breakdown:
experiment_split
train             178908
test               22563
val                21757
official_valid       234
Name: count, dtype: int64


In [8]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("/content/drive/MyDrive/DP-MMFL")

# Handle dataset folder casing variations
raw_candidates = [
    PROJECT_ROOT / "data" / "raw" / "CheXpertPlus",
    PROJECT_ROOT / "data" / "raw" / "chexpert_plus"
]
DATASET_ROOT = next((p for p in raw_candidates if p.exists()), raw_candidates[0])

CSV_PATH = DATASET_ROOT / "df_chexpert_plus_240401.csv"
V2_PATH = PROJECT_ROOT / "data" / "processed" / "chexpert_plus_manifest_v2.parquet"
V3_PATH = PROJECT_ROOT / "data" / "processed" / "chexpert_plus_manifest.parquet"

print(f"Reading CSV from: {CSV_PATH}")
print(f"Reading V2 from:  {V2_PATH}")

manifest = pd.read_parquet(V2_PATH)
original = pd.read_csv(CSV_PATH)

print(f"V2 Manifest Shape: {manifest.shape}")
print(f"Original CSV Shape: {original.shape}")

Reading CSV from: /content/drive/MyDrive/DP-MMFL/data/raw/chexpert_plus/df_chexpert_plus_240401.csv
Reading V2 from:  /content/drive/MyDrive/DP-MMFL/data/processed/chexpert_plus_manifest_v2.parquet
V2 Manifest Shape: (223462, 51)
Original CSV Shape: (223462, 27)


In [9]:
#Slice only image view and temporal order attributes
view_metadata = original[
    [
        "path_to_image",
        "frontal_lateral",
        "ap_pa",
        "patient_report_date_order",
    ]
].copy()

# Integrity assertions
assert not view_metadata["path_to_image"].duplicated().any(), "Duplicate image paths found in metadata slice."
assert set(manifest["path_to_image"]) == set(view_metadata["path_to_image"]), "Path key set mismatch between V2 and original CSV."

print("View metadata join key verified (1:1 correspondence confirmed).")

View metadata join key verified (1:1 correspondence confirmed).


In [10]:
# Merge view fields
manifest = manifest.merge(
    view_metadata,
    on="path_to_image",
    how="left",
    validate="one_to_one",
)

print("--- Missing Values in Appended Columns ---")
print(
    manifest[
        [
            "frontal_lateral",
            "ap_pa",
            "patient_report_date_order",
        ]
    ].isna().sum()
)

print("\n--- Projection (Frontal / Lateral) Distribution ---")
print(manifest["frontal_lateral"].value_counts(dropna=False))

print("\n--- Orientation (AP / PA) Distribution ---")
print(manifest["ap_pa"].value_counts(dropna=False))

--- Missing Values in Appended Columns ---
frontal_lateral                  0
ap_pa                        32391
patient_report_date_order        0
dtype: int64

--- Projection (Frontal / Lateral) Distribution ---
frontal_lateral
Frontal    191071
Lateral     32391
Name: count, dtype: int64

--- Orientation (AP / PA) Distribution ---
ap_pa
AP     161622
NaN     32391
PA      29432
LL         16
RL          1
Name: count, dtype: int64


In [12]:
print("\n--- Experiment Split Distribution Post-Merge ---")
split_counts = manifest["experiment_split"].value_counts()
print(split_counts)

# Assert total rows and partition counts
assert len(manifest) == 223462, f"Expected 223,462 rows, got {len(manifest)}"
assert split_counts["train"] == 178908, f"Train count mismatch: {split_counts.get('train')}"
assert split_counts["test"] == 22563, f"Test count mismatch: {split_counts.get('test')}"
assert split_counts["val"] == 21757, f"Val count mismatch: {split_counts.get('val')}"
assert split_counts["official_valid"] == 234, f"Official valid count mismatch: {split_counts.get('official_valid')}"

print("\nPartition consistency verified: Split counts match expectations exactly.")


--- Experiment Split Distribution Post-Merge ---
experiment_split
train             178908
test               22563
val                21757
official_valid       234
Name: count, dtype: int64

Partition consistency verified: Split counts match expectations exactly.


In [13]:
# Overwrite the root canonical manifest with the enriched data
manifest.to_parquet(V3_PATH, index=False)
print(f"Final manifest saved to: {V3_PATH}")

# Reload to ensure parquet serialization succeeded
check = pd.read_parquet(V3_PATH)
print("\n--- Reloaded Parquet Verification ---")
print(f"Final shape: {check.shape}")
print(check["experiment_split"].value_counts())

assert len(check) == 223462, "Reload length mismatch."
assert set(["frontal_lateral", "ap_pa", "patient_report_date_order"]).issubset(check.columns), "New metadata columns missing from persisted parquet."

print("\nFinal manifest validation passed successfully.")

Final manifest saved to: /content/drive/MyDrive/DP-MMFL/data/processed/chexpert_plus_manifest.parquet

--- Reloaded Parquet Verification ---
Final shape: (223462, 54)
experiment_split
train             178908
test               22563
val                21757
official_valid       234
Name: count, dtype: int64

Final manifest validation passed successfully.


In [14]:
%cd /content/DP-MMFL
!git status

/content/DP-MMFL
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
